# Setting up the STM32 to stream data over UART

**Topics:**
* Transmitting raw sample data
* UART communication using DMA

**End goal:**
1. Being able to sample an input signal on the STM32 at $f_s = 10 \text{KHz}$ and transmitting all the raw samples to be analyzed on the PC
2. Being able to analyze the recieved "real-world" signal on your PC using Python etc.



### Before you start:

In this assignment we will use the setup from assignment $6$ as a starting point, keeping the same hardware peripheral settings as when we implemented the Fast Fourier Transform. To recap, the main elements which need to be in place are:

1. [Configuring `Timer 2` to output an "update event" every $8\ 400$ clock cycles ($100 \mu s$), then configuring `ADC 1` to perform an A/D conversion each time an "update event" occurs before writing the converted sample to a buffer array using DMA](https://github.com/kaierih/AIS2201-Exercises/blob/master/2025_Exercise_6/2_Device_config.ipynb)
3. [Setting up a double buffer program architecture using `C` arrays and ISR callback functions](https://github.com/kaierih/AIS2201-Exercises/blob/master/2025_Exercise_6/3_Buffer_management.ipynb)

Make sure these components are in place before proceeding with the assignment! <br>
*PS: You can remove any code from `main.c` which is tied to calcualting the `FFT`. We will not be using the Fast Fourier Transform library in this assignment.*


### Introduction to UART and DMA

Something which is very useful when developing an embedded DSP system (as you will need to do in part 3 of the portfolio project) is having the ability to transmit signal information from the microprocessor to the PC so it can be analyzed using all the graphical tools available in eg. Pytohn or MATLAB. In previous assignments we have transmitted data from the STM32 to the PC using the STM32 equivalent of an Arduino's `Serial.print` function, that being [HAL_USART_Transmit](figures/HAL_USART_Transmit.png). One possible problem with using this function is that it transmits data in "blocking mode", meaning it waits for the serial transmission to complete (or exits due to a timeput) before resuming execution of the program. This can be fine for sending intermittent short status updates over UART, but when transmitting large amounts of data it can hold up the program much longer than is acceptable. 

Luckily, we don't actually need to wait for the data to finish transmitting before resuming the program. All we need to do is store the information we wish to transmit into an array, enable a DMA stream so the UART peripheral can read directly from that output buffer array, and enable interrupts so the peripheral trigger a callback function once the transmission is complete.



## a) *hardware configuration*

To be able to stream large amounts of data over UART using DMA, open the "Device Configuration Tool" and make the following changes under `Pinout & Configuration` $\rightarrow$ `Connectivity` $\rightarrow$ `USART2`:
* Under `Parameter Settings`, increase the baud rate to e.g. $460\ 800$ Bits/s. The main requirement here is that we have time to transmit $10\ 000$ samples with $32$ bits per sample (`float32`) in one second, also leaving some margin for metadata.
    * Example Screenshot!
* Under  `DMA Settings`, add a DMA stream for DMA request `USART2_TX`
    * Example Screenshot!
* Under `NVIC Settings`, make sure `DMA1 streamX global interrupt` and `USART2 global interrupt` is enabled.
    * Example Screenshot!

Once this is done select `Project` $\rightarrow$ `Generate Code`, or simply save configuration and select "Yes" when asked if you wish to generate code.

## b) *programming the STM to transmit sample buffer*

The intricacies of UART communication aren't really DSP curriculum, so our main goal now is to get a "working prototype" up and running by the easiest possible route. 

1. Attached to this assignment is a folder `uart_stream` which contains a `.h` file and a `.c` file. Add these to your project to the folders `Core/Inc` and `Core/Src` respectively.
2. Include the simple uart library in your `main.c` file by adding
>```C
>#include "uart_write_block.h"
>```
to the `Private includes` section
3. Create a `float` array named `TX_buffer` of size `WINDOW_SIZE`, and add a routine to  copy and convert integer data from the `ADC_buffer` to the new floating-point array in the main `while(1)` loop.
4. In the main `while(1)` loop, add the following code:
> ```C
> uart_write_block(&huart2, TX_buffer, WINDOW_SIZE); 
> ```

## c) *recieving data on your PC* 
Also attached to this assignment is a Python script `uart_to_file.py` we can use to recieve the necessary sample data and store it all in a file. Once your STM32 program is up and running (and hopefully streaming sample data over UART), hook it up to a Signal Generator utputting a $A=0.1\text{V}$ sinusoid and execute the python script to collect data to an output `.mat` file. 

Example usage:
> `python uart_to_file.py --port COM3 --fs 10000 --duration 10 --filename uart_capture.mat`
>
>Collects $10$ seconds of data assuming sample rate $f_s = 10 \text{KHz}$ from port `COM3` and stores to file `uart_capture.mat`

## d) *signal analysis*
Make any necessary modifications to the code cell below, load your collected signal data into python and analyze the signal using the itneractive SignalAnalyzer demo (or your own python script if that is your preference). Verify that the recieved signal is indeed a sinsusoid with the appropriate frequency. Also, take particular note of the following:

1. The sinusoid's power density level (from the plot)
2. The "noise floor" level (from the plot)
    * With no other noise sources present, the main contributor to signal noise will be *quantization noise*

In [ ]:
from demos import SignalAnalyzer
from scipy.io import loadmat
import numpy as np
%matplotlib widget

data = loadmat("uart_capture.mat", squeeze_me=True)
fs = data['fs']
xn = data['data']

xn = xn - np.mean(xn) # Remove DC component from signal

SignalAnalyzer(xn, f_s=fs)


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>
